In [ ]:
import pandas as pd
import geopandas as gpd  # Assuming your df is a GeoDataFrame; if not, load with gpd.read_file or similar


df = pd.read_csv('/home/dataopske/Desktop/jav/data/raw/porvertylevel/poverty_level.csv')

# Your df is assumed to be named 'df' with columns including 'ward' and geometry
# Replace 'df' with your actual DataFrame name if different
# df = your_dataframe_here  # e.g., wards_map or pd.read_csv(...); ensure it's a GeoDataFrame for .explore()

# Step 1: Updated ward-to-constituency mapping (completed with missing wards)
ward_to_constituency = {
    # Kibra Constituency (26.3% poverty)
    'Makina': 'Kibra',
    'Lindi': 'Kibra',
    'Laini Saba': 'Kibra',
    'Soweto East': 'Kibra',
    'Silanga': 'Kibra',
    'Sarang\'ombe': 'Kibra',  # Added
    'Woodley/kenyatta Golf Course': 'Kibra',  # Added (cleaned version)
    # Mathare Constituency (14.5% poverty)
    'Mathare North': 'Mathare',
    'Hospital': 'Mathare',
    'Kiamaiko': 'Mathare',
    'Huruma': 'Mathare',
    'Mabatini': 'Mathare',  # Added
    'Ngei': 'Mathare',  # Added
    'Mlango Kubwa': 'Mathare',  # Added
    # Westlands Constituency (17.9% poverty)
    'Karura': 'Westlands',
    'Parklands/Highridge': 'Westlands',
    'Kitisuru': 'Westlands',
    'Mountain View': 'Westlands',
    'Kangemi': 'Westlands',
    # Dagoretti North Constituency (12.7% poverty)
    'Kilimani': 'Dagoretti North',
    'Kawangware': 'Dagoretti North',
    'Gatina': 'Dagoretti North',
    'Kileleshwa': 'Dagoretti North',
    'Kabiro': 'Dagoretti North',
    # Dagoretti South Constituency (18.5% poverty)
    'Mutu-ini': 'Dagoretti South',
    'Ngando': 'Dagoretti South',
    'Riruta': 'Dagoretti South',
    'Uthiru/Ruthimitu': 'Dagoretti South',
    'Waithaka': 'Dagoretti South',
    # Langata Constituency (12.7% poverty)
    'Karen': 'Langata',
    'Nairobi West': 'Langata',
    'Mugumo-ini': 'Langata',
    'South C': 'Langata',
    'Nyayo Highrise': 'Langata',
    # Starehe Constituency (20.0% poverty)
    'Nairobi Central': 'Starehe',
    'Ngara': 'Starehe',
    'Pangani': 'Starehe',
    'Ziwani/Kariokor': 'Starehe',
    'Landimawe': 'Starehe',
    'Nairobi South': 'Starehe',
    # Kamukunji Constituency (9.3% poverty)
    'Pumwani': 'Kamukunji',
    'Eastleigh North': 'Kamukunji',
    'Eastleigh South': 'Kamukunji',
    'Airbase': 'Kamukunji',
    'California': 'Kamukunji',
    # Makadara Constituency (7.3% poverty)
    'Maringo/Hamza': 'Makadara',
    'Viwandani': 'Makadara',
    'Harambee': 'Makadara',
    'Makongeni': 'Makadara',
    # Kasarani Constituency (11.0% poverty)
    'Clay City': 'Kasarani',
    'Mwiki': 'Kasarani',
    'Kasarani': 'Kasarani',
    'Njiru': 'Kasarani',
    'Ruai': 'Kasarani',
    # Ruaraka Constituency (11.1% poverty)
    'Baba Dogo': 'Ruaraka',
    'Utalii': 'Ruaraka',
    'Mathare North': 'Ruaraka',  # Note: Overlap with Mathare; you may need to decide based on your data
    'Lucky Summer': 'Ruaraka',
    'Korogocho': 'Ruaraka',
    # Roysambu Constituency (12.5% poverty)
    'Roysambu': 'Roysambu',
    'Kahawa West': 'Roysambu',
    'Zimmerman': 'Roysambu',
    'Githurai': 'Roysambu',
    'Kahawa': 'Roysambu',
    # Embakasi South Constituency (13.8% poverty)
    'Imara Daima': 'Embakasi South',
    'Kwa Njenga': 'Embakasi South',
    'Kwa Reuben': 'Embakasi South',
    'Pipeline': 'Embakasi South',
    'Kware': 'Embakasi South',
    # Embakasi North Constituency (7.5% poverty)
    'Kariobangi North': 'Embakasi North',
    'Kariobangi South': 'Embakasi North',
    'Dandora Area I': 'Embakasi North',
    'Dandora Area II': 'Embakasi North',
    'Dandora Area III': 'Embakasi North',
    'Dandora Area IV': 'Embakasi North',
    # Embakasi Central Constituency (7.0% poverty)
    'Kayole North': 'Embakasi Central',
    'Kayole Central': 'Embakasi Central',
    'Kayole South': 'Embakasi Central',
    'Komarock': 'Embakasi Central',
    'Matopeni/Spring Valley': 'Embakasi Central',
    # Embakasi East Constituency (7.2% poverty)
    'Upper Savannah': 'Embakasi East',
    'Lower Savannah': 'Embakasi East',
    'Embakasi': 'Embakasi East',
    'Utawala': 'Embakasi East',
    'Mihango': 'Embakasi East',
    # Embakasi West Constituency (6.4% poverty - LOWEST)
    'Umoja I': 'Embakasi West',
    'Umoja II': 'Embakasi West',
    'Mowlem': 'Embakasi West',
    'Kariobangi South': 'Embakasi West',
}

# Step 1.5: Function to clean ward names to match dictionary keys
def clean_ward(ward):
    if pd.isna(ward):
        return ward
    ward = str(ward).replace(' Ward', '').replace(' Ward Ward', '')
    # Roman numerals for Dandora
    ward = ward.replace('Ii', 'II').replace('Iii', 'III').replace('Iv', 'IV')
    # Spelling fixes
    ward = ward.replace('Savanna', 'Savannah')
    # Case and slash fixes
    ward = ward.replace('Parklands/highridge', 'Parklands/Highridge')
    ward = ward.replace('Uthiru/ruthimitu', 'Uthiru/Ruthimitu')
    ward = ward.replace('Matopeni/spring Valley', 'Matopeni/Spring Valley')
    ward = ward.replace('Maringo/hamza', 'Maringo/Hamza')
    ward = ward.replace('Ziwani/kariokor', 'Ziwani/Kariokor')
    ward = ward.replace('Babandogo', 'Baba Dogo')
    ward = ward.replace(' Ii', ' II')
    ward = ward.replace(' IIi', ' III')  # In case of any odd casing
    ward = ward.replace("Sarang'ombe", "Sarang'ombe")  # Keep apostrophe
    ward = ward.replace('Woodley/kenyatta Golf Course', 'Woodley/kenyatta Golf Course')
    ward = ward.replace('Mlango Kubwa', 'Mlango Kubwa')
    ward = ward.replace('Mabatini', 'Mabatini')
    ward = ward.replace('Ngei', 'Ngei')
    return ward

# Apply cleaning and mapping
df['clean_ward'] = df['ward'].apply(clean_ward)
df['constituency'] = df['clean_ward'].map(ward_to_constituency)

# Step 2: Create poverty data DataFrame (uncomment and adjust if you have the CSV)
# poverty_lvl = pd.read_csv('nairobi_poverty_rates.csv')

# Or create from known rates
poverty_data = {
    'constituency': ['Kibra', 'Mathare', 'Westlands', 'Dagoretti North', 'Dagoretti South', 'Langata', 
                     'Starehe', 'Kamukunji', 'Makadara', 'Kasarani', 'Ruaraka', 'Roysambu', 
                     'Embakasi South', 'Embakasi North', 'Embakasi Central', 'Embakasi East', 'Embakasi West'],
    'poverty_rate': [26.3, 14.5, 17.9, 12.7, 18.5, 12.7, 20.0, 9.3, 7.3, 11.0, 11.1, 12.5, 13.8, 7.5, 7.0, 7.2, 6.4]
}
poverty_lvl = pd.DataFrame(poverty_data)

# Merge poverty data
df = df.merge(poverty_lvl, on='constituency', how='left')

# Step 3: Classify income based on poverty
def classify_by_poverty(poverty_rate):
    if pd.isna(poverty_rate):
        return 'Unknown'
    elif poverty_rate >= 18:
        return 'Low Income'
    elif poverty_rate >= 11:
        return 'Middle Income'
    else:
        return 'High Income'

df['income_class'] = df['poverty_rate'].apply(classify_by_poverty)

# Step 4: Create the map with poverty rate visualization (assumes df is GeoDataFrame)
m_poverty = df.explore(
    column='poverty_rate',
    cmap='RdYlGn_r',  # Red = high poverty, Green = low poverty
    legend=True,
    legend_kwds={'caption': 'Poverty Rate (%)'},
    tooltip=['ward', 'constituency', 'poverty_rate', 'income_class'],
    popup=['ward', 'constituency', 'poverty_rate', 'income_class', 'population'],  # Assumes 'population' column exists
    style_kwds={'color': 'black', 'weight': 0.5},
    tiles='CartoDB positron',
    name='Poverty Rates by Ward'
)

m_poverty  # In Jupyter, this displays the map